Creating gold_layer tables: `dim_date`, `dim_hour`, `dim_company` in `group3_gp.gold` using data from `group3_gp.silver.combined_taxi_trips`.

In [0]:
from pyspark.sql.functions import when, col, sequence, explode, lit, monotonically_increasing_id, to_date, current_date, year, month, date_format, dayofmonth, dayofweek, quarter, weekofyear

In [0]:
start_date = "2008-01-01"

dim_date = (
    spark.range(1)
    .select(sequence(to_date(lit(start_date)), current_date()).alias("d"))
    .select(explode("d").alias("date"))
    .select(
        monotonically_increasing_id().alias("date_id"),
        col("date"),
        year("date").alias("year"),
        month("date").alias("month"),
        date_format("date", "MMMM").alias("month_name"),
        dayofmonth("date").alias("day"),
        date_format("date", "EEEE").alias("day_of_week_name"),
        dayofweek("date").alias("day_of_week_num"),
        when(dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
        quarter("date").alias("quarter"),
        weekofyear("date").alias("week_of_year"),
        when(
            (col("date") >= lit("2020-03-01")) & (col("date") <= lit("2022-05-31")),
            True
        ).otherwise(False).alias("is_covid")
    )
    .orderBy("date")
)

dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.gold.dim_date")

In [0]:
# Create a DataFrame with hours 0-23
dim_hour = (
    spark.range(24)
    .withColumnRenamed("id", "hour_id")
    .withColumn(
        "period_of_day",
        when((col("hour_id") >= 5) & (col("hour_id") < 12), "morning")
        .when((col("hour_id") >= 12) & (col("hour_id") < 14), "noon")
        .when((col("hour_id") >= 14) & (col("hour_id") < 18), "afternoon")
        .when((col("hour_id") >= 18) & (col("hour_id") < 22), "evening")
        .otherwise("night")
    )
    .withColumn(
        "is_rush_traffic",
        when((col("hour_id").between(7, 9)) | (col("hour_id").between(16, 18)), True).otherwise(False)
    )
    .orderBy("hour_id")
)

dim_hour.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.gold.dim_hour")

In [0]:
%sql
DROP TABLE IF EXISTS group3_gp.gold.dim_company;

CREATE OR REPLACE TABLE group3_gp.gold.dim_company AS
SELECT
    dense_rank() OVER (ORDER BY service_type, taxi_type) AS company_id,
    service_type,
    taxi_type
FROM (
    SELECT DISTINCT
        service_type,
        taxi_type
    FROM group3_gp.silver.combined_taxi_trips
) t;